# 03 - Evaluation

This notebook compares the distilled student against the teacher and the reference responses on held-out prompts. The metrics are intentionally lightweight so they can run quickly during experimentation, while the saved samples still support manual review.

In [ ]:
from __future__ import annotations

from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer


In [ ]:
@dataclass(frozen=True)
class EvalConfig:
    teacher_model: str = "gpt2-medium"
    student_fallback_model: str = "distilgpt2"
    student_dir: Path = Path("../checkpoints/student")
    prepared_dir: Path = Path("../artifacts/prepared")
    report_dir: Path = Path("../reports")
    max_examples: int = 24
    max_prompt_tokens: int = 256
    max_new_tokens: int = 96
    num_beams: int = 3


config = EvalConfig()
config.report_dir.mkdir(parents=True, exist_ok=True)
asdict(config)


In [ ]:
def select_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def read_jsonl(path: Path) -> list[dict]:
    with path.open() as handle:
        return [json.loads(line) for line in handle if line.strip()]


def load_causal_lm(model_or_path: str | Path, device: torch.device):
    tokenizer = AutoTokenizer.from_pretrained(model_or_path, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_or_path)
    model.to(device)
    model.eval()
    return model, tokenizer


In [ ]:
def normalize_text(text: str) -> list[str]:
    return " ".join(text.lower().split()).split()


def token_f1(reference: str, prediction: str) -> float:
    ref_tokens = normalize_text(reference)
    pred_tokens = normalize_text(prediction)
    if not ref_tokens and not pred_tokens:
        return 1.0
    if not ref_tokens or not pred_tokens:
        return 0.0
    overlap = sum((Counter(ref_tokens) & Counter(pred_tokens)).values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)


def summarize_predictions(rows: list[dict], key: str) -> dict[str, float]:
    scores = [token_f1(row["reference"], row[key]) for row in rows]
    lengths = [len(normalize_text(row[key])) for row in rows]
    return {
        "token_f1": sum(scores) / len(scores),
        "avg_output_tokens": sum(lengths) / len(lengths),
        "exact_match": sum(row["reference"].strip() == row[key].strip() for row in rows) / len(rows),
    }


def pairwise_token_f1(rows: list[dict], left_key: str, right_key: str) -> float:
    return sum(token_f1(row[left_key], row[right_key]) for row in rows) / len(rows)


def generation_review_frame(rows: list[dict]) -> pd.DataFrame:
    review_rows = []
    for row in rows:
        reference_tokens = normalize_text(row["reference"])
        student_tokens = normalize_text(row["student"])
        teacher_tokens = normalize_text(row["teacher"])
        review_rows.append(
            {
                "id": row["id"],
                "category": row["category"],
                "reference_tokens": len(reference_tokens),
                "teacher_tokens": len(teacher_tokens),
                "student_tokens": len(student_tokens),
                "student_reference_length_ratio": len(student_tokens) / max(len(reference_tokens), 1),
                "student_teacher_length_ratio": len(student_tokens) / max(len(teacher_tokens), 1),
                "student_token_f1": token_f1(row["reference"], row["student"]),
                "student_teacher_token_f1": token_f1(row["teacher"], row["student"]),
                "empty_student_output": len(student_tokens) == 0,
                "prompt": row["prompt"],
                "student": row["student"],
            }
        )
    return pd.DataFrame(review_rows)


In [ ]:
@torch.inference_mode()
def generate_responses(model, tokenizer, prompts: list[str], device: torch.device) -> list[str]:
    outputs = []
    for prompt in tqdm(prompts, desc="generate"):
        encoded = tokenizer(
            prompt,
            truncation=True,
            max_length=config.max_prompt_tokens,
            return_tensors="pt",
        ).to(device)
        generated = model.generate(
            **encoded,
            max_new_tokens=config.max_new_tokens,
            num_beams=config.num_beams,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        text = tokenizer.decode(generated[0][encoded["input_ids"].shape[1] :], skip_special_tokens=True)
        outputs.append(" ".join(text.split()))
    return outputs


In [ ]:
device = select_device()
student_source = config.student_dir if (config.student_dir / "config.json").exists() else config.student_fallback_model

teacher, teacher_tokenizer = load_causal_lm(config.teacher_model, device)
student, student_tokenizer = load_causal_lm(student_source, device)

records = read_jsonl(config.prepared_dir / "validation.jsonl")[: config.max_examples]
prompts = [row["prompt"] for row in records]

{"device": str(device), "examples": len(records), "student_source": str(student_source)}


In [ ]:
teacher_outputs = generate_responses(teacher, teacher_tokenizer, prompts, device)
student_outputs = generate_responses(student, student_tokenizer, prompts, device)

comparison_rows = []
for row, teacher_text, student_text in zip(records, teacher_outputs, student_outputs):
    comparison_rows.append(
        {
            "id": row["id"],
            "category": row.get("category", ""),
            "prompt": row["prompt"],
            "reference": row["response"],
            "teacher": teacher_text,
            "student": student_text,
        }
    )

(config.report_dir / "generation_comparisons.json").write_text(json.dumps(comparison_rows, indent=2) + "\n")
review_frame = generation_review_frame(comparison_rows)
review_frame.to_csv(config.report_dir / "generation_review.csv", index=False)
review_frame[["id", "category", "reference_tokens", "student_tokens", "student_token_f1", "student_teacher_token_f1", "empty_student_output"]].head()


In [ ]:
metrics = {
    "teacher": summarize_predictions(comparison_rows, "teacher"),
    "student": summarize_predictions(comparison_rows, "student"),
}
metrics["student"]["teacher_token_f1"] = pairwise_token_f1(comparison_rows, "teacher", "student")
(config.report_dir / "evaluation_metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")

metric_frame = pd.DataFrame(metrics).T
ax = metric_frame[["token_f1", "exact_match"]].plot(kind="bar", figsize=(7, 4))
ax.set_ylim(0, 1)
ax.set_title("Held-out generation metrics")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(config.report_dir / "generation_metrics.png", dpi=160)
metric_frame
